In [1]:
import numpy as np
import pandas as pd
import json
from datetime import datetime
from tqdm import tqdm
tqdm.pandas()

In [2]:
review_df = pd.read_parquet("../data/output/review_data.parquet")
review_df = review_df.reset_index(names="review_id")
review_df

,review_id,asin,parent_asin,user_id,review_datetime,review_title,review_text,review_images,verified_purchase,helpful_vote,review_rating
0,0,b07bfs3g7p,b0bqsk9qcf,agci7fah4gl5fi65hylkwtmfz2cq,2019-07-03,malware,mcaffee is malware,[],False,0,1.0
1,1,b00ctq6sig,b00ctq6sig,ahspldnw5oouk2plh7gxlacfbznq,2015-02-16,lots of fun,i love playing tapped out because it is fun to...,[],True,0,5.0
2,2,b0066wjlu6,b0066wjlu6,ahspldnw5oouk2plh7gxlacfbznq,2013-03-04,light up the dark,i love this flashlight app! it really illumin...,[],True,0,5.0
3,3,b00kcymawk,b00kcymawk,ah6catodivpvuojewhrsrcskaoha,2019-06-20,fun game,one of my favorite games,[],True,0,4.0
4,4,b00p1rk566,b00p1rk566,aeiny4xoinmmjck5gz3m6mmhbn6a,2014-12-11,i am not that good at it but my kids are,cute game. i am not that good at it but my kid...,[],True,0,4.0
...,...,...,...,...,...,...,...,...,...,...,...
4880176,4880176,b0763n3mvw,b0763n3mvw,agokecumai3w6mc7a7kbfbjvpn7q,2020-12-25,gog,very fun and addictive and exciting,[],True,0,5.0
4880177,4880177,b00et56y48,b00et56y48,ah6amundosz2sgdidmysre4rnmdq,2021-01-06,worst game ever,worst game ever toxic people and bad connectio...,[],True,1,1.0
4880178,4880178,b009zkspdk,b009zkspdk,ah4pj73qn75ajm5vsct53aoadcga,2013-05-28,better!!!,this fabulous game is 10000 times better than ...,[],True,2,5.0
4880179,4880179,b00m9j3ioa,b00m9j3ioa,ahqetdykhvdngmgbwvjl6vjxjgfq,2016-09-03,it has everything i need and more,awesome! i upgraded from coreldraw 8. i was wo...,[],True,0,5.0


In [3]:
meta_df = pd.read_parquet("../data/output/meta_data.parquet")
meta_df

,parent_asin,item_title,main_category,categories,description,features,details,item_images,item_videos,item_rating,store,price
0,b00vrpsgeo,accupressure guide,appstore for android,[],[Acupressure technique is very ancient and ver...,[All the pressing point has been explained wit...,"{""Release Date"": ""2015"", ""Date first listed on...","{'hi_res': [None, None, None, None], 'large': ...","{'title': [], 'url': [], 'user_id': []}",NaN,mappsguru,0.00
1,b00nwqxxhq,ankylosaurus fights back - smithsonian's prehi...,appstore for android,[],[Join Ankylosaurus in this interactive book ap...,[ENCOURAGE literacy skills with highlighted na...,"{""Release Date"": ""2014"", ""Date first listed on...","{'hi_res': [None, None, None, None, None], 'la...","{'title': [], 'url': [], 'user_id': []}",NaN,"oceanhouse media, inc",2.99
2,b00rfkp6ac,mahjong 2015,appstore for android,[],[Mahjong 2015 is a free solitaire matching gam...,[Mahjong 2015 is a free solitaire matching gam...,"{""Release Date"": ""2014"", ""Date first listed on...","{'hi_res': [None, None, None], 'large': ['http...","{'title': [], 'url': [], 'user_id': []}",NaN,sophiathach,0.00
3,b00sp2qu0e,jewels brick breakout,appstore for android,[],[Jewels Brick Breakout is a glowing jewels bri...,"[Game Features:, - Intuitive touch controls wi...","{""Release Date"": ""2015"", ""Date first listed on...","{'hi_res': [None, None, None, None, None, None...","{'title': [], 'url': [], 'user_id': []}",NaN,bad chicken,0.00
4,b01dzit64o,traffic police: off-road cub,appstore for android,[],[Become the best road police officer in Cube C...,"[In this game you will find:, - Killer police ...","{""Release Date"": ""2016"", ""Date first listed on...","{'hi_res': [None, None, None, None], 'large': ...","{'title': [], 'url': [], 'user_id': []}",NaN,dast 2 for metro,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
89246,b006kwvrgi,how to start a commercial loan broker plus bus...,software,"[business & marketing plans, business & office...",[The How to Start a Commercial Loan Brokerage ...,[Everything You Need to Know About Starting a ...,"{""Is Discontinued By Manufacturer"": ""No"", ""Ite...","{'hi_res': [None, None, None, None, None], 'la...","{'title': [], 'url': [], 'user_id': []}",2.0,howtostartabusinessdb,29.95
89247,b000j54pra,norton uninstall deluxe,software,"[software, utilities]",[The safe uninstaller from the makers of Norto...,[Safely removes programs and files you no long...,"{""Is Discontinued By Manufacturer"": ""No"", ""Pac...","{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",2.0,symantec,NaN
89248,b00mg0ohb0,beginning pro tools le 8 [instant access],None,"[education & reference, software]",[This video takes you from start to finish on ...,[],"{""Language"": ""English"", ""Item model number"": ""...","{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",3.0,alfred music,15.95
89249,b001gn6yuk,tell me more spanish performance version 9 (2 ...,None,"[education & reference, languages, software]","[From the Manufacturer, This award-winning lan...",[Spanish language-learning software with 2 lev...,"{""Language"": ""English, French"", ""Package Dimen...","{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",2.0,auralog,209.00


In [4]:
def count_review_images(image_list):
    """Safely count number of images"""
    if isinstance(image_list, list):
        return len(image_list)
    elif pd.isna(image_list):
        return np.nan
    else:
        # If it's not a list (maybe a string or other type), try to convert
        try:
            if isinstance(image_list, str):
                import ast
                parsed = ast.literal_eval(image_list)
                return len(parsed) if isinstance(parsed, list) else 1
            else:
                return 1
        except:
            return 0

def count_item_img(img_dict):
    hi_res = img_dict.get("hi_res", [])
    large = img_dict.get("large", [])
    thumb = img_dict.get("thumb", [])

    return (
        sum(1 for x in hi_res if x) +
        sum(1 for x in large if x) +
        sum(1 for x in thumb if x)
    )

def count_item_videos(video_dict):
    url_list = video_dict.get("url", [])
    return len(url_list)

def count_item_variant(img_dict):
    variant = img_dict.get("variant", [])
    return len(variant)

def get_release_datetime(detail_str):
    detail_dict = json.loads(detail_str)
    date_str = detail_dict.get("Date First Available", [])

    if date_str == []:
        return np.nan

    return datetime.strptime(date_str, "%B %d, %Y")

def get_country_manufacturer(detail_str):
    detail_dict = json.loads(detail_str)
    manu_str = detail_dict.get("Manufacturer", [])

    if manu_str == []:
        return np.nan

    return str(manu_str).strip().lower()

def get_country_origin(detail_str):
    detail_dict = json.loads(detail_str)
    origin_str = detail_dict.get("Country of Origin", [])

    if origin_str == []:
        return np.nan

    return str(origin_str).strip().lower()

merged_df = (
    review_df
    .merge(
        meta_df,
        how="left",
        on="parent_asin"
    )
    .dropna(subset=["price"])
)

merged_df["num_review_img"] = merged_df["review_images"].apply(count_review_images)
merged_df["num_item_img"] = merged_df["item_images"].apply(count_item_img)
merged_df["num_item_videos"] = merged_df["item_videos"].apply(count_item_videos)
merged_df["num_item_variant"] = merged_df["item_images"].apply(count_item_variant)

# merged_df["item_release_datetime"] = merged_df["details"].apply(get_release_datetime)
# merged_df["item_manufacturer"] = merged_df["details"].apply(get_country_manufacturer)
# merged_df["item_country_origin"] = merged_df["details"].apply(get_country_origin)

merged_df = merged_df.explode("categories").rename(columns={"categories": "category"})

merged_df

,review_id,asin,parent_asin,user_id,review_datetime,review_title,review_text,review_images,verified_purchase,helpful_vote,...,details,item_images,item_videos,item_rating,store,price,num_review_img,num_item_img,num_item_videos,num_item_variant
0,0,b07bfs3g7p,b0bqsk9qcf,agci7fah4gl5fi65hylkwtmfz2cq,2019-07-03,malware,mcaffee is malware,[],False,0,...,"{""Product Dimensions"": ""7.5 x 5.5 x 0.5 inches...",{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['McAfee REAL Support', 'How to Acti...",343.0,mcafee,34.99,0,21,10,7
0,0,b07bfs3g7p,b0bqsk9qcf,agci7fah4gl5fi65hylkwtmfz2cq,2019-07-03,malware,mcaffee is malware,[],False,0,...,"{""Product Dimensions"": ""7.5 x 5.5 x 0.5 inches...",{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['McAfee REAL Support', 'How to Acti...",343.0,mcafee,34.99,0,21,10,7
0,0,b07bfs3g7p,b0bqsk9qcf,agci7fah4gl5fi65hylkwtmfz2cq,2019-07-03,malware,mcaffee is malware,[],False,0,...,"{""Product Dimensions"": ""7.5 x 5.5 x 0.5 inches...",{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['McAfee REAL Support', 'How to Acti...",343.0,mcafee,34.99,0,21,10,7
1,1,b00ctq6sig,b00ctq6sig,ahspldnw5oouk2plh7gxlacfbznq,2015-02-16,lots of fun,i love playing tapped out because it is fun to...,[],True,0,...,"{""Release Date"": ""2013"", ""Date first listed on...","{'hi_res': [None, None, None, None, None, None...","{'title': [], 'url': [], 'user_id': []}",19370.0,electronic arts,0.00,0,6,0,6
2,2,b0066wjlu6,b0066wjlu6,ahspldnw5oouk2plh7gxlacfbznq,2013-03-04,light up the dark,i love this flashlight app! it really illumin...,[],True,0,...,"{""Release Date"": ""2011"", ""Date first listed on...","{'hi_res': [None, None, None, None], 'large': ...","{'title': [], 'url': [], 'user_id': []}",7859.0,smallte.ch,0.00,0,4,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4880175,4880175,b07th23gwv,b07th23gwv,afxn4h3e55alrifjpsiqfyoxklha,2022-03-26,amazing game with very little flaws,it is a really fun game,[],True,2,...,"{""Release Date"": ""2019"", ""Date first listed on...","{'hi_res': [None, None, None, None, None, None...","{'title': [], 'url': ['https://images-na.ssl-i...",281.0,computer lunch,0.00,0,7,1,7
4880176,4880176,b0763n3mvw,b0763n3mvw,agokecumai3w6mc7a7kbfbjvpn7q,2020-12-25,gog,very fun and addictive and exciting,[],True,0,...,"{""Release Date"": ""2017"", ""Date first listed on...","{'hi_res': [None, None, None, None, None, None...","{'title': [], 'url': ['https://images-na.ssl-i...",367.0,北京雷灵网络科技有限公司,0.00,0,10,1,10
4880177,4880177,b00et56y48,b00et56y48,ah6amundosz2sgdidmysre4rnmdq,2021-01-06,worst game ever,worst game ever toxic people and bad connectio...,[],True,1,...,"{""Release Date"": ""2013"", ""Date first listed on...","{'hi_res': [None, None, None, None, None, None...","{'title': [], 'url': [], 'user_id': []}",7793.0,ninja kiwi,0.00,0,6,0,6
4880178,4880178,b009zkspdk,b009zkspdk,ah4pj73qn75ajm5vsct53aoadcga,2013-05-28,better!!!,this fabulous game is 10000 times better than ...,[],True,2,...,"{""Release Date"": ""2012"", ""Date first listed on...","{'hi_res': [None, None, None, None, None, None...","{'title': [], 'url': [], 'user_id': []}",5605.0,candy rufus games,3.99,0,9,0,9


In [5]:
user_df = (
    merged_df[merged_df["verified_purchase"] == True]
    .groupby(["user_id", "main_category", "category"])
    .agg({
        "review_id": "count",
        "review_rating": "mean",
        "review_datetime": "max",
        # "helpful_vote": "sum",
        # "num_review_img": "sum"
    })
    .reset_index()
    .rename(columns={
        "review_id": "num_review_given",
        "review_rating": "avg_rating_given",
        "review_datetime": "latest_user_datetime"
    })
)

user_df

,user_id,main_category,category,num_review_given,avg_rating_given,latest_user_datetime
0,ae222h3fgxwlhrfumgms2rr57ndq,software,antivirus,1,5.0,2021-04-19
1,ae222h3fgxwlhrfumgms2rr57ndq,software,antivirus & security,1,5.0,2021-04-19
2,ae222h3fgxwlhrfumgms2rr57ndq,software,software,1,5.0,2021-04-19
3,ae223tlcv274eett6yjjoj4imcza,software,business & office,1,5.0,2013-08-15
4,ae223tlcv274eett6yjjoj4imcza,software,linux,1,5.0,2013-08-15
...,...,...,...,...,...,...
247611,ahzzy3ar4ogdfqgsassvc2rv2s6a,software,software,1,5.0,2021-05-23
247612,ahzzymxrskktc5k5wx6qnieuusma,software,clip art,1,5.0,2020-11-09
247613,ahzzymxrskktc5k5wx6qnieuusma,software,home publishing,1,5.0,2020-11-09
247614,ahzzymxrskktc5k5wx6qnieuusma,software,lifestyle & hobbies,1,5.0,2020-11-09


In [8]:
item_df = (
    merged_df
    .groupby([
        "parent_asin",
        "main_category", "category",
        "num_item_variant",
        "num_item_img", "num_item_videos"
    ])
    .agg({
        "user_id": "count",
        "review_datetime": "max",
        "review_rating": "mean",
        "price": "mean"
    })
    .rename(columns={
        "user_id": "num_review_received",
        "review_datetime": "latest_review_datetime",
        "review_rating": "avg_rating_received",
        "price": "avg_price"
    })
    .reset_index()
)

item_df

,parent_asin,main_category,category,num_item_variant,num_item_img,num_item_videos,num_review_received,latest_review_datetime,avg_rating_received,avg_price
0,0077563387,software,courses,1,3,0,2,2017-03-07,1.000,29.99
1,0077563387,software,software,1,3,0,2,2017-03-07,1.000,29.99
2,0439026008,software,brands,10,29,0,3,2009-04-28,3.000,14.65
3,0439026008,software,oregon trail,10,29,0,3,2009-04-28,3.000,14.65
4,0439026008,software,software,10,29,0,3,2009-04-28,3.000,14.65
...,...,...,...,...,...,...,...,...,...,...
7930,b0cbn6ygz3,software,office suites,6,12,0,2,2019-07-14,4.000,49.99
7931,b0cbn6ygz3,software,software,6,12,0,2,2019-07-14,4.000,49.99
7932,b0cc7jglms,software,antivirus & security,10,30,2,8,2023-05-17,4.875,348.00
7933,b0cc7jglms,software,internet security suites,10,30,2,8,2023-05-17,4.875,348.00
